# MKP Biased Initialization — Experiment Notebook

## 0) Enviornment Setup
*Run cells 0a thru 0d once at the start of session*

In [ ]:
# ============================================================
# IMPORTS — run this first, every session - 0a
# ============================================================
import json
import time
import random
import inspect
import os
import numpy as np
import torch
import json, time, random
import numpy as np
from google.colab import files

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("Imports done.")

PyTorch version : 2.10.0+cpu
CUDA available  : False
Imports done.


In [ ]:
# ============================================================
# CORE FUNCTIONS — run once per session - 0b
# ============================================================

# ── Feasibility ──────────────────────────────────────────────
def is_feasible(sol, weights, capacities, n, m):
    for i in range(m):
        total = sum(weights[i][j] * sol[j] for j in range(n))
        if total > capacities[i]:
            return False
    return True

# ── Fitness (raw value, no penalty) ─────────────────────────
def fitness(sol, values, n):
    return sum(values[j] * sol[j] for j in range(n))

# ── Efficiency scores for repair ────────────────────────────
def compute_efficiency_scores(values, weights, capacities, n, m):
    scores = []
    for j in range(n):
        denom = sum(weights[i][j] / capacities[i] for i in range(m))
        scores.append(values[j] / denom if denom > 0 else 0)
    return scores

# ── Repair operator ──────────────────────────────────────────
def repair(sol, values, weights, capacities, efficiency_scores, n, m):
    sol = list(sol).copy()
    order = sorted(range(n), key=lambda j: efficiency_scores[j])
    for j in order:
        if is_feasible(sol, weights, capacities, n, m):
            break
        sol[j] = 0
    for j in reversed(order):
        if sol[j] == 0:
            sol[j] = 1
            if not is_feasible(sol, weights, capacities, n, m):
                sol[j] = 0
    return sol

# ── Initialization ───────────────────────────────────────────
def initialize_population(population_size, n):
    return [[random.randint(0, 1) for _ in range(n)]
            for _ in range(population_size)]

def initialize_population_biased(population_size, n, p_j_values):
    return [[1 if random.random() < p_j_values[j] else 0
             for j in range(n)]
            for _ in range(population_size)]

def initialize_population_hybrid(population_size, n, p_j_values,
                                  biased_ratio):
    n_biased  = int(biased_ratio * population_size)
    n_uniform = population_size - n_biased
    biased    = initialize_population_biased(n_biased, n, p_j_values)
    uniform   = initialize_population(n_uniform, n)
    return biased + uniform

# ── Tournament selection ─────────────────────────────────────
def tournament_select(population, fitness_scores):
    a = random.randint(0, len(population) - 1)
    b = random.randint(0, len(population) - 1)
    if fitness_scores[a] >= fitness_scores[b]:
        return population[a]
    else:
        return population[b]

# ── Uniform crossover ────────────────────────────────────────
def uniform_crossover(parent1, parent2, n):
    return [parent1[j] if random.random() < 0.5 else parent2[j]
            for j in range(n)]

# ── Mutation ─────────────────────────────────────────────────
def mutate(sol, n, rate=None):
    rate = rate if rate is not None else 1 / n
    return [1 - sol[j] if random.random() < rate else sol[j]
            for j in range(n)]

# ── Sanity check ─────────────────────────────────────────────
def sanity_check(instances):
    for alpha, inst in instances.items():
        w = np.array(inst['weights'])
        assert w.shape == (M, N), f"Wrong shape for alpha={alpha}: {w.shape}"
        print(f"  alpha={alpha}: weights shape={w.shape} confirmed")

print("Core functions defined.")

Core functions defined.


In [ ]:
# ============================================================
# GA FUNCTIONS — run once per session - 0c
# ============================================================

# ── Diversity measurement ────────────────────────────────────
def mean_hamming_distance(population, n):
    total = 0
    count = 0
    for i in range(len(population)):
        for j in range(i + 1, len(population)):
            total += sum(population[i][k] != population[j][k]
                        for k in range(n))
            count += 1
    return total / count if count > 0 else 0.0


# ── Repair-based GA (used for reference best finder) ─────────
def run_ga(n, m, values, weights, capacities,
           population_size=50,
           max_offspring=10000,
           best_known=None,
           target_pct=0.99,
           init_method='uniform',
           p_j_values=None,
           hybrid_ratio=None,
           seed=None,
           track_every=50):

    if seed is not None:
        random.seed(seed)

    efficiency_scores = compute_efficiency_scores(
        values, weights, capacities, n, m
    )

    if init_method == 'hybrid' and hybrid_ratio is not None:
        population = initialize_population_hybrid(
            population_size, n, p_j_values, hybrid_ratio
        )
    elif init_method == 'biased':
        population = initialize_population_biased(
            population_size, n, p_j_values
        )
    else:
        population = initialize_population(population_size, n)

    feasible_before_repair = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    for idx in range(population_size):
        population[idx] = repair(
            population[idx], values, weights,
            capacities, efficiency_scores, n, m
        )

    fitness_scores   = [fitness(sol, values, n) for sol in population]
    avg_fitness_gen0 = sum(fitness_scores) / population_size
    best_fitness     = max(fitness_scores)
    best_solution    = population[fitness_scores.index(best_fitness)].copy()
    diversity_gen0   = mean_hamming_distance(population, n)

    convergence_point = None
    if best_known and best_fitness >= best_known * target_pct:
        convergence_point = 0

    curve           = [(0, best_fitness, avg_fitness_gen0, diversity_gen0)]
    offspring_count = 0

    while offspring_count < max_offspring:
        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)
        child   = repair(child, values, weights,
                         capacities, efficiency_scores, n, m)

        child_fitness = fitness(child, values, n)
        worst_idx     = fitness_scores.index(min(fitness_scores))

        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        if (convergence_point is None and best_known and
                best_fitness >= best_known * target_pct):
            convergence_point = offspring_count

        if offspring_count % track_every == 0:
            avg_div = mean_hamming_distance(population, n)
            curve.append((offspring_count, best_fitness,
                          sum(fitness_scores) / population_size,
                          avg_div))

    return {
        'feasibility_rate_gen0': feasible_before_repair,
        'avg_fitness_gen0':      avg_fitness_gen0,
        'diversity_gen0':        diversity_gen0,
        'best_fitness':          best_fitness,
        'convergence_point':     convergence_point,
        'gap_pct': ((best_known - best_fitness) / best_known * 100
                    if best_known else None),
        'curve':                 curve
    }


# ── Death penalty GA ─────────────────────────────────────────
def run_ga_death_penalty(n, m, values, weights, capacities,
                          population_size=50,
                          max_offspring=20000,
                          best_known=None,
                          target_pct=0.99,
                          init_method='uniform',
                          p_j_values=None,
                          hybrid_ratio=None,
                          seed=None,
                          track_every=50):

    if seed is not None:
        random.seed(seed)

    if init_method == 'hybrid' and hybrid_ratio is not None:
        population = initialize_population_hybrid(
            population_size, n, p_j_values, hybrid_ratio
        )
    elif init_method == 'biased':
        population = initialize_population_biased(
            population_size, n, p_j_values
        )
    else:
        population = initialize_population(population_size, n)

    feasible_before = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    def death_fitness(sol):
        if not is_feasible(sol, weights, capacities, n, m):
            return 0
        return sum(values[j] * sol[j] for j in range(n))

    fitness_scores   = [death_fitness(sol) for sol in population]
    avg_fitness_gen0 = sum(fitness_scores) / population_size
    best_fitness     = max(fitness_scores)
    best_solution    = population[fitness_scores.index(best_fitness)].copy()
    pct_of_best_gen0 = (best_fitness / best_known * 100
                        if best_known and best_fitness > 0 else 0.0)
    diversity_gen0   = mean_hamming_distance(population, n)

    convergence      = {t: None for t in CONVERGENCE_THRESHOLDS}
    curve            = [(0, best_fitness, avg_fitness_gen0, diversity_gen0)]
    offspring_count  = 0

    while offspring_count < max_offspring:
        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)

        child_fitness = death_fitness(child)
        worst_idx     = fitness_scores.index(min(fitness_scores))

        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        if best_known and is_feasible(best_solution, weights,
                                       capacities, n, m):
            for t in CONVERGENCE_THRESHOLDS:
                if convergence[t] is None and \
                        best_fitness >= best_known * (t / 100):
                    convergence[t] = offspring_count

        if offspring_count % track_every == 0:
            avg_div = mean_hamming_distance(population, n)
            curve.append((offspring_count, best_fitness,
                          sum(fitness_scores) / population_size,
                          avg_div))

    best_feasible = is_feasible(best_solution, weights,
                                 capacities, n, m)

    return {
        'feasibility_rate_gen0':  feasible_before,
        'avg_fitness_gen0':       avg_fitness_gen0,
        'diversity_gen0':         diversity_gen0,
        'pct_of_best_gen0':       pct_of_best_gen0,
        'best_fitness_final':     best_fitness,
        'best_solution_feasible': best_feasible,
        'pct_of_best_final':      (best_fitness / best_known * 100
                                   if best_known and best_feasible
                                   else 0.0),
        'convergence':            convergence,
        'curve':                  curve
    }


# ── Soft penalty GA ──────────────────────────────────────────
def run_ga_penalty_fitness(n, m, values, weights, capacities,
                            population_size=50,
                            max_offspring=20000,
                            best_known=None,
                            target_pct=0.99,
                            init_method='uniform',
                            p_j_values=None,
                            hybrid_ratio=None,
                            penalty_multiplier=5.0,
                            seed=None,
                            track_every=50):

    if seed is not None:
        random.seed(seed)

    if init_method == 'hybrid' and hybrid_ratio is not None:
        population = initialize_population_hybrid(
            population_size, n, p_j_values, hybrid_ratio
        )
    elif init_method == 'biased':
        population = initialize_population_biased(
            population_size, n, p_j_values
        )
    else:
        population = initialize_population(population_size, n)

    feasible_before = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    def pfitness(sol):
        total_value = sum(values[j] * sol[j] for j in range(n))
        penalty = 0
        for i in range(m):
            total_weight = sum(weights[i][j] * sol[j] for j in range(n))
            violation    = max(0, total_weight - capacities[i])
            penalty     += penalty_multiplier * violation
        return total_value - penalty

    fitness_scores   = [pfitness(sol) for sol in population]
    avg_fitness_gen0 = sum(fitness_scores) / population_size
    best_fitness     = max(fitness_scores)
    best_solution    = population[fitness_scores.index(best_fitness)].copy()
    pct_of_best_gen0 = (best_fitness / best_known * 100
                        if best_known and best_fitness > 0 else 0.0)
    diversity_gen0   = mean_hamming_distance(population, n)

    convergence      = {t: None for t in CONVERGENCE_THRESHOLDS}
    curve            = [(0, best_fitness, avg_fitness_gen0, diversity_gen0)]
    offspring_count  = 0

    while offspring_count < max_offspring:
        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)

        child_fitness = pfitness(child)
        worst_idx     = fitness_scores.index(min(fitness_scores))

        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        if best_known and is_feasible(best_solution, weights,
                                       capacities, n, m):
            for t in CONVERGENCE_THRESHOLDS:
                if convergence[t] is None and \
                        best_fitness >= best_known * (t / 100):
                    convergence[t] = offspring_count

        if offspring_count % track_every == 0:
            avg_div = mean_hamming_distance(population, n)
            curve.append((offspring_count, best_fitness,
                          sum(fitness_scores) / population_size,
                          avg_div))

    best_feasible = is_feasible(best_solution, weights,
                                 capacities, n, m)

    return {
        'feasibility_rate_gen0':  feasible_before,
        'avg_fitness_gen0':       avg_fitness_gen0,
        'diversity_gen0':         diversity_gen0,
        'pct_of_best_gen0':       pct_of_best_gen0,
        'best_fitness_final':     best_fitness,
        'best_solution_feasible': best_feasible,
        'pct_of_best_final':      (best_fitness / best_known * 100
                                   if best_known and best_feasible
                                   else 0.0),
        'convergence':            convergence,
        'curve':                  curve
    }

print("GA functions defined.")

GA functions defined.


In [16]:
# ============================================================
# SUMMARY FUNCTIONS — run once per session 0d
# ============================================================
def summarize_comparison(results_uni, results_bia, label, ref):

    def thresh_stats(results, t):
        vals = [r['convergence'][t] for r in results
                if r['convergence'].get(t) is not None]
        if not vals:
            return None, 0.0
        return np.mean(vals), len(vals) / len(results)

    def fmt(val):
        return f"{val:.0f}" if val is not None else "never"

    def improvement(u, b):
        if u is None or b is None:
            return "N/A"
        return f"{(u - b) / u * 100:+.1f}%"

    feas0_u  = np.mean([r['feasibility_rate_gen0'] for r in results_uni])
    feas0_b  = np.mean([r['feasibility_rate_gen0'] for r in results_bia])
    fit0_u   = np.mean([r['pct_of_best_gen0']      for r in results_uni])
    fit0_b   = np.mean([r['pct_of_best_gen0']      for r in results_bia])
    fitf_u   = np.mean([r['pct_of_best_final']     for r in results_uni])
    fitf_b   = np.mean([r['pct_of_best_final']     for r in results_bia])
    div0_u   = np.mean([r['diversity_gen0']         for r in results_uni])
    div0_b   = np.mean([r['diversity_gen0']         for r in results_bia])
    divf_u   = np.mean([r['curve'][-1][3] for r in results_uni
                        if r['curve'] and len(r['curve'][-1]) > 3])
    divf_b   = np.mean([r['curve'][-1][3] for r in results_bia
                        if r['curve'] and len(r['curve'][-1]) > 3])

    print(f"\n{'='*65}")
    print(f"  SUMMARY — {label}")
    print(f"{'='*65}")
    print(f"  Reference best : {ref}")
    print(f"  {'Metric':<45} {'Uniform':>9} {'Biased':>9}")
    print(f"  {'-'*63}")
    print(f"  {'Feasibility rate at gen 0':<45} {feas0_u:>9.3f} {feas0_b:>9.3f}")
    print(f"  {'Fitness at gen 0 (% of ref)':<45} {fit0_u:>8.1f}% {fit0_b:>8.1f}%")
    print(f"  {'Final fitness (% of ref)':<45} {fitf_u:>8.1f}% {fitf_b:>8.1f}%")
    print(f"  {'Diversity at gen 0 (avg Hamming)':<45} {div0_u:>9.2f} {div0_b:>9.2f}")
    print(f"  {'Diversity at end (avg Hamming)':<45} {divf_u:>9.2f} {divf_b:>9.2f}")
    print()

    for t in CONVERGENCE_THRESHOLDS:
        c_u, r_u = thresh_stats(results_uni, t)
        c_b, r_b = thresh_stats(results_bia, t)
        print(f"  {f'Offspring to {t}%':<45} {fmt(c_u):>9} {fmt(c_b):>9}")
        print(f"    {'improvement':<43} {improvement(c_u, c_b):>19}")

    print()
    for t in CONVERGENCE_THRESHOLDS:
        c_u, r_u = thresh_stats(results_uni, t)
        c_b, r_b = thresh_stats(results_bia, t)
        print(f"  Reached {t}%"
              f"  Uniform={int(r_u*len(results_uni))}/{len(results_uni)}"
              f"  Biased={int(r_b*len(results_bia))}/{len(results_bia)}")

    print(f"\n  {'-'*63}")
    print(f"  RESULTS LOG")
    print(f"  Experiment : {label}")
    print(f"  Reference  : {ref}")
    print(f"  Feas @ 0   : Uniform={feas0_u:.1%}  Biased={feas0_b:.1%}")
    print(f"  Fit  @ 0   : Uniform={fit0_u:.1f}%  Biased={fit0_b:.1f}%")
    print(f"  Div  @ 0   : Uniform={div0_u:.2f}  Biased={div0_b:.2f}")
    print(f"  Div  @ end : Uniform={divf_u:.2f}  Biased={divf_b:.2f}")
    for t in CONVERGENCE_THRESHOLDS:
        c_u, _ = thresh_stats(results_uni, t)
        c_b, _ = thresh_stats(results_bia, t)
        print(f"  Conv to {t}% : Uniform={fmt(c_u)}  Biased={fmt(c_b)}")
    print(f"  Final qual : Uniform={fitf_u:.1f}%  Biased={fitf_b:.1f}%")
    print(f"{'='*65}\n")


def summarize_hybrid_sweep(hybrid_results, ref):

    def thresh_mean(results, t):
        vals = [r['convergence'].get(t) for r in results
                if r['convergence'].get(t) is not None]
        if not vals:
            return 'never'
        return f"{np.mean(vals):.0f} ({len(vals)}/{len(results)})"

    def div_final(results):
        vals = [r['curve'][-1][3] for r in results
                if r['curve'] and len(r['curve'][-1]) > 3]
        return np.mean(vals) if vals else 0.0

    ratios = sorted(hybrid_results.keys())

    print(f"\n{'='*75}")
    print(f"  HYBRID SWEEP SUMMARY — alpha=0.25, death penalty")
    print(f"  Reference best: {ref}")
    print(f"{'='*75}")

    # header
    header = f"  {'Metric':<35} "
    for r in ratios:
        lbl = f"r={r:.2f}"
        header += f"{lbl:<13}"
    print(header)
    print(f"  {'-'*73}")

    # feasibility at gen 0
    row = f"  {'Feasibility at gen 0':<35} "
    for ratio in ratios:
        v = f"{np.mean([r['feasibility_rate_gen0'] for r in hybrid_results[ratio]]):.1%}"
        row += f"{v:<13}"
    print(row)

    # diversity at gen 0
    row = f"  {'Diversity at gen 0 (Hamming)':<35} "
    for ratio in ratios:
        v = f"{np.mean([r['diversity_gen0'] for r in hybrid_results[ratio]]):.1f}"
        row += f"{v:<13}"
    print(row)

    # diversity at end
    row = f"  {'Diversity at end (Hamming)':<35} "
    for ratio in ratios:
        v = f"{div_final(hybrid_results[ratio]):.1f}"
        row += f"{v:<13}"
    print(row)

    # final fitness
    row = f"  {'Final fitness (% of ref)':<35} "
    for ratio in ratios:
        v = f"{np.mean([r['pct_of_best_final'] for r in hybrid_results[ratio]]):.1f}%"
        row += f"{v:<13}"
    print(row)

    print()

    # convergence thresholds
    for t in CONVERGENCE_THRESHOLDS:
        row = f"  {'Offspring to '+str(t)+'%':<35} "
        for ratio in ratios:
            v = thresh_mean(hybrid_results[ratio], t)
            row += f"{v:<13}"
        print(row)

    print()

    # reached counts
    for t in CONVERGENCE_THRESHOLDS:
        row = f"  {'Reached '+str(t)+'%':<35} "
        for ratio in ratios:
            results = hybrid_results[ratio]
            vals = [r['convergence'].get(t) for r in results
                    if r['convergence'].get(t) is not None]
            v = f"{len(vals)}/{len(results)}"
            row += f"{v:<13}"
        print(row)

    print(f"{'='*75}\n")

def summarize_penalty_sweep(penalty_results, alpha, ref):

    penalties = list(penalty_results.keys())

    def fmt_val(val):
        return f"{val:.0f}" if val is not None else "never"

    def thresh_mean(results, t):
        vals = [r['convergence'].get(t) for r in results
                if r['convergence'].get(t) is not None]
        if not vals:
            return f"never"
        return f"{np.mean(vals):.0f}({len(vals)}/{len(results)})"

    def div_final(results):
        vals = [r['curve'][-1][3] for r in results
                if r['curve'] and len(r['curve'][-1]) > 3]
        return np.mean(vals) if vals else 0.0

    def reached(results, t):
        vals = [r['convergence'].get(t) for r in results
                if r['convergence'].get(t) is not None]
        return f"{len(vals)}/{len(results)}"

    col_w = 16
    label_w = 35

    print(f"\n{'='*120}")
    print(f"  PENALTY SWEEP SUMMARY — alpha={alpha}, uniform vs biased")
    print(f"  Reference best: {ref}")
    print(f"{'='*120}")

    # header
    header = f"  {'Metric':<{label_w}} {'Init':<8}"
    for p in penalties:
        lbl = f"x{p}" if p != 'death' else "death"
        header += f"{lbl:<{col_w}}"
    print(header)
    print(f"  {'-'*118}")

    # helper to print one metric row for both uniform and biased
    def print_row(metric_label, fn_uni, fn_bia):
        row_u = f"  {metric_label:<{label_w}} {'uniform':<8}"
        row_b = f"  {'':<{label_w}} {'biased':<8}"
        for p in penalties:
            row_u += f"{fn_uni(penalty_results[p]['uniform']):<{col_w}}"
            row_b += f"{fn_bia(penalty_results[p]['biased']):<{col_w}}"
        print(row_u)
        print(row_b)
        print()

    print_row(
        "Feasibility at gen 0",
        lambda r: f"{np.mean([x['feasibility_rate_gen0'] for x in r]):.1%}",
        lambda r: f"{np.mean([x['feasibility_rate_gen0'] for x in r]):.1%}"
    )
    print_row(
        "Diversity at gen 0 (Hamming)",
        lambda r: f"{np.mean([x['diversity_gen0'] for x in r]):.1f}",
        lambda r: f"{np.mean([x['diversity_gen0'] for x in r]):.1f}"
    )
    print_row(
        "Diversity at end (Hamming)",
        lambda r: f"{div_final(r):.1f}",
        lambda r: f"{div_final(r):.1f}"
    )
    print_row(
        "Final fitness (% of ref)",
        lambda r: f"{np.mean([x['pct_of_best_final'] for x in r]):.1f}%",
        lambda r: f"{np.mean([x['pct_of_best_final'] for x in r]):.1f}%"
    )

    for t in CONVERGENCE_THRESHOLDS:
        print_row(
            f"Offspring to {t}%",
            lambda r, t=t: thresh_mean(r, t),
            lambda r, t=t: thresh_mean(r, t)
        )

    for t in [98, 99]:
        print_row(
            f"Reached {t}%",
            lambda r, t=t: reached(r, t),
            lambda r, t=t: reached(r, t)
        )

    print(f"{'='*120}\n")

def summarize_tightness_sweep(tightness_results):

    alphas  = sorted(tightness_results.keys())
    methods = ['uniform', 'biased', 'hybrid']
    col_w   = 14
    label_w = 38

    def mean_field(results, field):
        vals = [r[field] for r in results
                if r.get(field) is not None]
        return np.mean(vals) if vals else 0.0

    def thresh_mean(results, t):
        vals = [r['convergence'].get(t) for r in results
                if r['convergence'].get(t) is not None]
        if not vals:
            return 'never'
        return f"{np.mean(vals):.0f}({len(vals)}/{len(results)})"

    def div_final(results):
        vals = [r['curve'][-1][3] for r in results
                if r['curve'] and len(r['curve'][-1]) > 3]
        return np.mean(vals) if vals else 0.0

    print(f"\n{'='*120}")
    print(f"  TIGHTNESS SWEEP SUMMARY — death penalty | "
          f"uniform vs biased vs 50% hybrid")
    print(f"{'='*120}")

    # header
    header = f"  {'Metric':<{label_w}} {'Method':<10}"
    for alpha in alphas:
        header += f"{'a='+str(alpha):<{col_w}}"
    print(header)
    print(f"  {'-'*118}")

    def print_metric(metric_label, fn):
        for method in methods:
            row = f"  {metric_label if method=='uniform' else '':<{label_w}} {method:<10}"
            for alpha in alphas:
                if method in tightness_results[alpha]:
                    row += f"{fn(tightness_results[alpha][method]):<{col_w}}"
                else:
                    row += f"{'N/A':<{col_w}}"
            print(row)
        print()

    print_metric(
        "Feasibility at gen 0",
        lambda r: f"{mean_field(r,'feasibility_rate_gen0'):.1%}"
    )
    print_metric(
        "Diversity at gen 0 (Hamming)",
        lambda r: f"{mean_field(r,'diversity_gen0'):.1f}"
    )
    print_metric(
        "Diversity at end (Hamming)",
        lambda r: f"{div_final(r):.1f}"
    )
    print_metric(
        "Final fitness (% of ref)",
        lambda r: f"{mean_field(r,'pct_of_best_final'):.1f}%"
    )

    for t in CONVERGENCE_THRESHOLDS:
        print_metric(
            f"Offspring to {t}%",
            lambda r, t=t: thresh_mean(r, t)
        )

    for t in [90, 95, 98]:
        print_metric(
            f"Reached {t}%",
            lambda r, t=t: f"{sum(1 for x in r if x['convergence'].get(t) is not None)}/{len(r)}"
        )

    print(f"{'='*120}\n")

print("Tightness sweep summary function defined.")

print("Summary functions defined.")

Tightness sweep summary function defined.
Summary functions defined.


## 1) Instance Generation & Constants
*Run cell 1 once at the start of session - locked paramters*

In [14]:
# ============================================================
# INSTANCE GENERATION — locked, never changes - 1
# ============================================================
import json, time, random
import numpy as np

N, M, INSTANCE_SEED      = 100, 3, 42
ALPHA_VALUES = [0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.75, 0.80, 0.90]
POP_SIZE                  = 50
N_RUNS                    = 20
N_OFFSPRING               = 20_000
TRACK_EVERY               = 50
PENALTY_MULT              = 5.0
PJ_SAMPLES                = 10_000_000
PJ_SEED                   = 42
REF_OFFSPRING             = 10_000
REF_RUNS                  = 10
REF_POP                   = 50
CONVERGENCE_THRESHOLDS    = [50, 60, 70, 80, 90, 95, 98, 99]

def generate_random_mkp_instance(n, m, alpha, seed=42):
    np.random.seed(seed)
    values     = np.random.randint(1, 100, n)
    weights    = np.random.randint(1,  50, (m, n))
    capacities = np.array([int(alpha * weights[i].sum()) for i in range(m)])
    return values, weights, capacities

instances = {}
for alpha in ALPHA_VALUES:
    v, w, c = generate_random_mkp_instance(N, M, alpha, seed=INSTANCE_SEED)
    instances[alpha] = {'values': v, 'weights': w, 'capacities': c}
    print(f"  α={alpha}: capacities={c}")

sanity_check(instances)
print("Instances generated and locked.")

  α=0.1: capacities=[252 244 280]
  α=0.2: capacities=[505 489 560]
  α=0.25: capacities=[632 611 700]
  α=0.3: capacities=[758 734 840]
  α=0.4: capacities=[1011  978 1120]
  α=0.5: capacities=[1264 1223 1400]
  α=0.6: capacities=[1517 1468 1680]
  α=0.7: capacities=[1770 1712 1959]
  α=0.75: capacities=[1896 1835 2100]
  α=0.8: capacities=[2023 1957 2240]
  α=0.9: capacities=[2276 2202 2520]
  alpha=0.1: weights shape=(3, 100) confirmed
  alpha=0.2: weights shape=(3, 100) confirmed
  alpha=0.25: weights shape=(3, 100) confirmed
  alpha=0.3: weights shape=(3, 100) confirmed
  alpha=0.4: weights shape=(3, 100) confirmed
  alpha=0.5: weights shape=(3, 100) confirmed
  alpha=0.6: weights shape=(3, 100) confirmed
  alpha=0.7: weights shape=(3, 100) confirmed
  alpha=0.75: weights shape=(3, 100) confirmed
  alpha=0.8: weights shape=(3, 100) confirmed
  alpha=0.9: weights shape=(3, 100) confirmed
Instances generated and locked.


## 2) Reference Best Finder
*Run cell 2 once when running notebook for the first time - loads from file if already computed*

In [15]:
# ============================================================
# REFERENCE BEST FINDER - 2
# repair-based GA, 100k offspring, 20 runs, take max
# loads from file if already computed — skips recompute
# ============================================================
REF_SAVE_PATH = '/content/reference_bests.json'

def find_reference_best(alpha, n_runs=REF_RUNS, max_offspring=REF_OFFSPRING):
    inst    = instances[alpha]
    v, w, c = inst['values'], inst['weights'], inst['capacities']
    best_overall = 0

    print(f"\nFinding reference best for α={alpha}...")
    for run in range(n_runs):
        r   = run_ga(N, M, v, w, c,
                     population_size=REF_POP,
                     max_offspring=max_offspring,
                     best_known=None,
                     init_method='uniform',
                     seed=run)
        val = r['best_fitness']
        if val > best_overall:
            best_overall = val
        print(f"  Run {run+1:02d}/{n_runs}: best={val:.0f}  "
              f"overall={best_overall:.0f}")

    print(f"  → Reference best for α={alpha}: {best_overall}")
    return int(best_overall)


if os.path.exists(REF_SAVE_PATH):
    # ── Load from file — skip recompute ──────────────────────
    with open(REF_SAVE_PATH, 'r') as f:
        ref_meta = json.load(f)
    reference_bests = {float(k): v for k, v in
                       ref_meta['reference_bests'].items()}
    print("✅ Reference bests loaded from file — skipping recompute:")
    for alpha, ref in reference_bests.items():
        print(f"   α={alpha}: {ref}")

else:
    # ── Compute fresh and save ────────────────────────────────
    reference_bests = {}
    for alpha in ALPHA_VALUES:
        reference_bests[alpha] = find_reference_best(alpha)

    ref_meta = {
        'reference_bests': {str(a): r for a, r in reference_bests.items()},
        'methodology': {
            'description': (
                'Best feasible fitness found by repair-based GA over multiple '
                'runs. Not a proven optimal — a well-resourced best-known value '
                'used as convergence target.'
            ),
            'ga_type':        'repair-based (run_ga)',
            'init_method':    'uniform',
            'population_size': REF_POP,
            'max_offspring':   REF_OFFSPRING,
            'n_runs':          REF_RUNS,
            'seeds':           list(range(REF_RUNS)),
            'selection':       'binary tournament k=2',
            'crossover':       'uniform',
            'mutation_rate':   f'1/n = {1/N:.4f}',
            'instance': {
                'n':              N,
                'm':              M,
                'seed':           INSTANCE_SEED,
                'values_range':   '1-99',
                'weights_range':  '1-49',
                'capacities':     'int(alpha * sum of constraint weights)'
            }
        }
    }

    with open(REF_SAVE_PATH, 'w') as f:
        json.dump(ref_meta, f, indent=2)

    print(f"\nReference bests saved → {REF_SAVE_PATH}")
    print("\nReference bests locked:")
    for alpha, ref in reference_bests.items():
        print(f"   α={alpha}: {ref}")


Finding reference best for α=0.1...
  Run 01/10: best=1150  overall=1150
  Run 02/10: best=1150  overall=1150
  Run 03/10: best=1130  overall=1150
  Run 04/10: best=1150  overall=1150
  Run 05/10: best=1146  overall=1150
  Run 06/10: best=1147  overall=1150
  Run 07/10: best=1150  overall=1150
  Run 08/10: best=1146  overall=1150
  Run 09/10: best=1150  overall=1150
  Run 10/10: best=1150  overall=1150
  → Reference best for α=0.1: 1150

Finding reference best for α=0.2...
  Run 01/10: best=1984  overall=1984
  Run 02/10: best=1986  overall=1986
  Run 03/10: best=1995  overall=1995
  Run 04/10: best=1995  overall=1995
  Run 05/10: best=1986  overall=1995
  Run 06/10: best=1987  overall=1995
  Run 07/10: best=1986  overall=1995
  Run 08/10: best=1995  overall=1995
  Run 09/10: best=1995  overall=1995
  Run 10/10: best=1982  overall=1995
  → Reference best for α=0.2: 1995

Finding reference best for α=0.25...
  Run 01/10: best=2367  overall=2367
  Run 02/10: best=2368  overall=2368
  Ru

##3) p_j Computation (GPU)
*Run cell 3 once per session*

In [17]:
# ============================================================
# p_j COMPUTATION — GPU Monte Carlo, 10M samples
# loads from file if already computed — skips recompute
# ============================================================
PJ_SAVE_PATH = '/content/pj_values.json'

def compute_pj_gpu(weights, capacities, n_samples=PJ_SAMPLES,
                   batch_size=50_000, seed=PJ_SEED):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {device}")

    m, n  = weights.shape
    W     = torch.tensor(weights,    dtype=torch.float32, device=device)
    C     = torch.tensor(capacities, dtype=torch.float32, device=device)
    torch.manual_seed(seed)

    count_in  = torch.zeros(n, device=device)
    count_out = torch.zeros(n, device=device)
    total_in  = torch.zeros(n, device=device)
    total_out = torch.zeros(n, device=device)

    processed = 0
    t0 = time.time()

    while processed < n_samples:
        bs    = min(batch_size, n_samples - processed)
        X     = torch.randint(0, 2, (bs, n),
                              device=device, dtype=torch.float32)
        usage = X @ W.T

        for j in range(n):
            in_j            = X[:, j] == 1
            out_j           = X[:, j] == 0
            usage_without_j = usage - X[:, j:j+1] * W[:, j]
            thresh_in       = C - W[:, j]
            feas_in         = (usage_without_j <= thresh_in).all(dim=1) & in_j
            feas_out        = (usage_without_j <= C        ).all(dim=1) & out_j
            count_in[j]    += feas_in.sum()
            count_out[j]   += feas_out.sum()
            total_in[j]    += in_j.sum()
            total_out[j]   += out_j.sum()

        processed += bs

    rho_in  = (count_in  / total_in.clamp(min=1)).cpu().numpy()
    rho_out = (count_out / total_out.clamp(min=1)).cpu().numpy()
    p_j     = rho_in / (rho_in + rho_out + 1e-12)

    print(f"  Done in {time.time()-t0:.1f}s | "
          f"p_j range [{p_j.min():.3f}, {p_j.max():.3f}] "
          f"mean={p_j.mean():.3f}")
    return p_j.tolist()


if os.path.exists(PJ_SAVE_PATH):
    with open(PJ_SAVE_PATH, 'r') as f:
        pj_meta = json.load(f)
    pj_values = {float(k): np.array(v)
                 for k, v in pj_meta['pj_values'].items()}

    # check if any new alphas need computing
    missing_alphas = [a for a in ALPHA_VALUES if a not in pj_values]

    if missing_alphas:
        print(f"Loading existing p_j, computing missing: {missing_alphas}")
        for alpha in missing_alphas:
            inst = instances[alpha]
            print(f"\nComputing p_j for alpha={alpha}...")
            pj_values[alpha] = np.array(
                compute_pj_gpu(inst['weights'], inst['capacities'])
            )
        # resave with new alphas merged in
        with open(PJ_SAVE_PATH, 'w') as f:
            json.dump({
                'pj_values': {str(a): pj.tolist()
                              for a, pj in pj_values.items()},
                'params': {
                    'n_samples': PJ_SAMPLES,
                    'seed': PJ_SEED,
                    'instance_seed': INSTANCE_SEED,
                    'n': N, 'm': M,
                    'alpha_values': ALPHA_VALUES,
                }
            }, f, indent=2)
        print(f"\nUpdated p_j saved -> {PJ_SAVE_PATH}")
    else:
        print("All p_j loaded from file — skipping recompute:")
        for alpha, pj in sorted(pj_values.items()):
            print(f"  alpha={alpha}: p_j range [{pj.min():.3f}, "
                  f"{pj.max():.3f}] mean={pj.mean():.3f}")

else:
    # compute all fresh
    pj_values = {}
    for alpha in ALPHA_VALUES:
        inst = instances[alpha]
        print(f"\nComputing p_j for alpha={alpha}...")
        pj_values[alpha] = np.array(
            compute_pj_gpu(inst['weights'], inst['capacities'])
        )

    with open(PJ_SAVE_PATH, 'w') as f:
        json.dump({
            'pj_values': {str(a): pj.tolist()
                          for a, pj in pj_values.items()},
            'params': {
                'n_samples': PJ_SAMPLES,
                'seed': PJ_SEED,
                'instance_seed': INSTANCE_SEED,
                'n': N, 'm': M,
                'alpha_values': ALPHA_VALUES,
            }
        }, f, indent=2)
    print(f"\np_j saved -> {PJ_SAVE_PATH}")

print("\np_j ready for all alpha values.")


Computing p_j for alpha=0.1...
  Device: cpu
  Done in 79.0s | p_j range [0.000, 0.000] mean=0.000

Computing p_j for alpha=0.2...
  Device: cpu
  Done in 78.6s | p_j range [0.000, 0.000] mean=0.000

Computing p_j for alpha=0.25...
  Device: cpu
  Done in 78.1s | p_j range [0.000, 0.000] mean=0.000

Computing p_j for alpha=0.3...
  Device: cpu
  Done in 78.9s | p_j range [0.145, 0.462] mean=0.292

Computing p_j for alpha=0.4...
  Device: cpu
  Done in 78.4s | p_j range [0.300, 0.484] mean=0.377

Computing p_j for alpha=0.5...
  Device: cpu
  Done in 77.9s | p_j range [0.414, 0.493] mean=0.449

Computing p_j for alpha=0.6...
  Device: cpu
  Done in 77.7s | p_j range [0.487, 0.499] mean=0.493

Computing p_j for alpha=0.7...
  Device: cpu
  Done in 77.4s | p_j range [0.500, 0.500] mean=0.500

Computing p_j for alpha=0.75...
  Device: cpu
  Done in 78.2s | p_j range [0.500, 0.500] mean=0.500

Computing p_j for alpha=0.8...
  Device: cpu
  Done in 79.7s | p_j range [0.500, 0.500] mean=0.50

## 4a) Experiment Functions
*Run cell 4a once per session — defines run_experiment and summaries*

In [ ]:
# ============================================================
# EXPERIMENT PIPELINE — function definition
# ============================================================
def make_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {str(k): make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_serializable(i) for i in obj]
    if isinstance(obj, tuple):
        return [make_serializable(i) for i in obj]
    return obj

def run_experiment(label, alpha, penalty_type,
                   init_method='uniform',
                   hybrid_ratio=None,
                   n_runs=N_RUNS):

    inst     = instances[alpha]
    v, w, c  = inst['values'], inst['weights'], inst['capacities']
    ref      = reference_bests[alpha]
    pj       = pj_values[alpha]
    ga_fn    = (run_ga_death_penalty if penalty_type == 'death'
                else run_ga_penalty_fitness)

    ratio_str = (f"  hybrid_ratio={hybrid_ratio}"
                 if init_method == 'hybrid' else "")

    print(f"\n{'='*65}")
    print(f"  EXPERIMENT: {label}")
    print(f"{'='*65}")
    print(f"  Instance       : n={N}, m={M}, seed={INSTANCE_SEED}")
    print(f"  Tightness      : alpha={alpha}")
    print(f"  Penalty        : {penalty_type}" +
          (f"  (multiplier={PENALTY_MULT})" if penalty_type != 'death' else ""))
    print(f"  Init method    : {init_method}{ratio_str}")
    print(f"  Runs           : {n_runs}")
    print(f"  Offspring/run  : {N_OFFSPRING}")
    print(f"  Population     : {POP_SIZE}")
    print(f"  Mutation rate  : 1/n = {1/N:.4f}")
    print(f"  Tournament     : binary (k=2)")
    print(f"  Crossover      : uniform")
    print(f"  Reference best : {ref}")
    print(f"  p_j samples    : {PJ_SAMPLES:,}")
    print(f"  Thresholds     : {CONVERGENCE_THRESHOLDS}")
    print(f"{'='*65}")

    results = []

    for run in range(n_runs):
        print(f"  Run {run+1:02d}/{n_runs}...", end=" ", flush=True)

        base_kwargs = dict(
            n=N, m=M,
            values=v, weights=w, capacities=c,
            population_size=POP_SIZE,
            max_offspring=N_OFFSPRING,
            best_known=ref,
            track_every=TRACK_EVERY,
            init_method=init_method,
            p_j_values=pj,
            hybrid_ratio=hybrid_ratio,
            seed=run * 7,
        )
        if penalty_type != 'death':
            base_kwargs['penalty_multiplier'] = PENALTY_MULT

        r = ga_fn(**base_kwargs)
        results.append(r)

        print(f"feas0={r['feasibility_rate_gen0']:.1%} "
              f"div0={r['diversity_gen0']:.1f} "
              f"final={r['pct_of_best_final']:.1f}%")

    save_key  = label.replace(" ","_").replace("=","").replace(".","")
    save_path = f'/content/results_{save_key}.json'
    with open(save_path, 'w') as f:
        json.dump(make_serializable({
            'label':        label,
            'alpha':        alpha,
            'penalty':      penalty_type,
            'init_method':  init_method,
            'hybrid_ratio': hybrid_ratio,
            'ref':          ref,
            'params': {
                'n': N, 'm': M, 'instance_seed': INSTANCE_SEED,
                'pop_size': POP_SIZE, 'n_runs': n_runs,
                'n_offspring': N_OFFSPRING,
                'mutation_rate': 1/N,
                'penalty_mult': PENALTY_MULT,
                'pj_samples': PJ_SAMPLES,
                'pj_seed': PJ_SEED,
                'ref_offspring': REF_OFFSPRING,
                'ref_runs': REF_RUNS,
                'tournament': 'binary k=2',
                'crossover': 'uniform',
                'convergence_thresholds': CONVERGENCE_THRESHOLDS,
            },
            'results': results,
        }), f, indent=2)
    print(f"\n  Saved -> {save_path}")

    return results


def run_experiment_comparison(label, alpha, penalty_type, n_runs=N_RUNS):
    results_uni = run_experiment(
        label + " uniform", alpha, penalty_type,
        init_method='uniform', n_runs=n_runs
    )
    results_bia = run_experiment(
        label + " biased", alpha, penalty_type,
        init_method='biased', n_runs=n_runs
    )
    summarize_comparison(results_uni, results_bia, label,
                         reference_bests[alpha])
    return results_uni, results_bia

print("run_experiment defined.")

run_experiment defined.


## 4b) Baseline Experiments — 6 Conditions
*Run cell 4b to reproduce baseline results*

In [ ]:
# ============================================================
# RUN ALL 6 CONDITIONS - 4b
# ============================================================
all_results = {}

conditions = [
    ("Death penalty α=0.25", 0.25, "death"),
    ("Death penalty α=0.50", 0.50, "death"),
    ("Death penalty α=0.75", 0.75, "death"),
    ("Penalty x5.0  α=0.25", 0.25, "penalty"),
    ("Penalty x5.0  α=0.50", 0.50, "penalty"),
    ("Penalty x5.0  α=0.75", 0.75, "penalty"),
]

for label, alpha, penalty_type in conditions:
    uni, bia = run_experiment(label, alpha, penalty_type)
    all_results[label] = (uni, bia)

print("\n✅ ALL BASELINE EXPERIMENTS COMPLETE")


  EXPERIMENT: Death penalty α=0.25
  Instance       : n=100, m=3, seed=42
  Tightness      : α=0.25
  Penalty        : death
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 2375
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... U feas0=0.0% B feas0=42.0% U final=0.0% B final=95.9%
  Run 02/20... U feas0=0.0% B feas0=52.0% U final=0.0% B final=96.5%
  Run 03/20... U feas0=0.0% B feas0=38.0% U final=0.0% B final=97.9%
  Run 04/20... U feas0=0.0% B feas0=40.0% U final=0.0% B final=98.0%
  Run 05/20... U feas0=0.0% B feas0=50.0% U final=0.0% B final=95.3%
  Run 06/20... U feas0=0.0% B feas0=48.0% U final=0.0% B final=96.5%
  Run 07/20... U feas0=0.0% B feas0=56.0% U final=0.0% B final=97.4%
  Run 08/20... U feas0=0.0% B feas0=46.0% U final=0.0% B final=95.3%
  Run 09/20... U feas0=0.0% B feas0=54.0% U final=0.0% B

## 4c) Hybrid Initialization Sweep
*Run cell 4c — alpha=0.25, death penalty, ratios 0/25/50/75/100*

In [ ]:
# ============================================================
# HYBRID SWEEP — alpha=0.25, death penalty 4c
# ============================================================
HYBRID_RATIOS = [0.0, 0.25, 0.50, 0.75, 1.0]

hybrid_results = {}

for ratio in HYBRID_RATIOS:
    if ratio == 0.0:
        init_method = 'uniform'
        label = "Hybrid sweep a=0.25 death ratio=0.00 uniform"
    elif ratio == 1.0:
        init_method = 'biased'
        label = "Hybrid sweep a=0.25 death ratio=1.00 biased"
    else:
        init_method = 'hybrid'
        label = f"Hybrid sweep a=0.25 death ratio={ratio:.2f}"

    results = run_experiment(
        label=label,
        alpha=0.25,
        penalty_type='death',
        init_method=init_method,
        hybrid_ratio=ratio if init_method == 'hybrid' else None,
        n_runs=N_RUNS
    )
    hybrid_results[ratio] = results

print("HYBRID SWEEP COMPLETE")
summarize_hybrid_sweep(hybrid_results, reference_bests[0.25])


  EXPERIMENT: Hybrid sweep a=0.25 death ratio=0.00 uniform
  Instance       : n=100, m=3, seed=42
  Tightness      : alpha=0.25
  Penalty        : death
  Init method    : uniform
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 2375
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... feas0=0.0% div0=49.6 final=0.0%
  Run 02/20... feas0=0.0% div0=49.9 final=0.0%
  Run 03/20... feas0=0.0% div0=50.2 final=0.0%
  Run 04/20... feas0=0.0% div0=50.0 final=0.0%
  Run 05/20... feas0=0.0% div0=50.1 final=0.0%
  Run 06/20... feas0=0.0% div0=50.0 final=0.0%
  Run 07/20... feas0=0.0% div0=50.1 final=0.0%
  Run 08/20... feas0=0.0% div0=50.2 final=0.0%
  Run 09/20... feas0=0.0% div0=50.1 final=0.0%
  Run 10/20... feas0=0.0% div0=50.2 final=0.0%
  Run 11/20... feas0=0.0% div0=50.1 final=0.0%
  Run 12/20... feas0=0.0% div0=49.9 

In [ ]:
# ============================================================
# EXPERIMENT PIPELINE — function definition
# ============================================================
def make_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {str(k): make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_serializable(i) for i in obj]
    if isinstance(obj, tuple):
        return [make_serializable(i) for i in obj]
    return obj

def run_experiment(label, alpha, penalty_type,
                   init_method='uniform',
                   hybrid_ratio=None,
                   n_runs=N_RUNS):

    inst     = instances[alpha]
    v, w, c  = inst['values'], inst['weights'], inst['capacities']
    ref      = reference_bests[alpha]
    pj       = pj_values[alpha]
    ga_fn    = (run_ga_death_penalty if penalty_type == 'death'
                else run_ga_penalty_fitness)

    ratio_str = (f"  hybrid_ratio={hybrid_ratio}"
                 if init_method == 'hybrid' else "")

    print(f"\n{'='*65}")
    print(f"  EXPERIMENT: {label}")
    print(f"{'='*65}")
    print(f"  Instance       : n={N}, m={M}, seed={INSTANCE_SEED}")
    print(f"  Tightness      : alpha={alpha}")
    print(f"  Penalty        : {penalty_type}" +
          (f"  (multiplier={PENALTY_MULT})" if penalty_type != 'death' else ""))
    print(f"  Init method    : {init_method}{ratio_str}")
    print(f"  Runs           : {n_runs}")
    print(f"  Offspring/run  : {N_OFFSPRING}")
    print(f"  Population     : {POP_SIZE}")
    print(f"  Mutation rate  : 1/n = {1/N:.4f}")
    print(f"  Tournament     : binary (k=2)")
    print(f"  Crossover      : uniform")
    print(f"  Reference best : {ref}")
    print(f"  p_j samples    : {PJ_SAMPLES:,}")
    print(f"  Thresholds     : {CONVERGENCE_THRESHOLDS}")
    print(f"{'='*65}")

    results = []

    for run in range(n_runs):
        print(f"  Run {run+1:02d}/{n_runs}...", end=" ", flush=True)

        base_kwargs = dict(
            n=N, m=M,
            values=v, weights=w, capacities=c,
            population_size=POP_SIZE,
            max_offspring=N_OFFSPRING,
            best_known=ref,
            track_every=TRACK_EVERY,
            init_method=init_method,
            p_j_values=pj,
            hybrid_ratio=hybrid_ratio,
            seed=run * 7,
        )
        if penalty_type != 'death':
            base_kwargs['penalty_multiplier'] = PENALTY_MULT

        r = ga_fn(**base_kwargs)
        results.append(r)

        print(f"feas0={r['feasibility_rate_gen0']:.1%} "
              f"div0={r['diversity_gen0']:.1f} "
              f"final={r['pct_of_best_final']:.1f}%")

    save_key  = label.replace(" ","_").replace("=","").replace(".","")
    save_path = f'/content/results_{save_key}.json'
    with open(save_path, 'w') as f:
        json.dump(make_serializable({
            'label':        label,
            'alpha':        alpha,
            'penalty':      penalty_type,
            'init_method':  init_method,
            'hybrid_ratio': hybrid_ratio,
            'ref':          ref,
            'params': {
                'n': N, 'm': M, 'instance_seed': INSTANCE_SEED,
                'pop_size': POP_SIZE, 'n_runs': n_runs,
                'n_offspring': N_OFFSPRING,
                'mutation_rate': 1/N,
                'penalty_mult': PENALTY_MULT,
                'pj_samples': PJ_SAMPLES,
                'pj_seed': PJ_SEED,
                'ref_offspring': REF_OFFSPRING,
                'ref_runs': REF_RUNS,
                'tournament': 'binary k=2',
                'crossover': 'uniform',
                'convergence_thresholds': CONVERGENCE_THRESHOLDS,
            },
            'results': results,
        }), f, indent=2)
    print(f"\n  Saved -> {save_path}")

    return results


def run_experiment_comparison(label, alpha, penalty_type, n_runs=N_RUNS):
    results_uni = run_experiment(
        label + " uniform", alpha, penalty_type,
        init_method='uniform', n_runs=n_runs
    )
    results_bia = run_experiment(
        label + " biased", alpha, penalty_type,
        init_method='biased', n_runs=n_runs
    )
    summarize_comparison(results_uni, results_bia, label,
                         reference_bests[alpha])
    return results_uni, results_bia

print("run_experiment defined.")

run_experiment defined.


## 5a) Penalty Sweep, alpha=0.25

In [ ]:
# ============================================================
# PENALTY SWEEP — alpha=0.25
# penalties: 5, 10, 25, 50, 100, 500, death
# uniform vs biased, 20 runs, 20k offspring
# ============================================================
PENALTY_VALUES = [5.0, 10.0, 25.0, 50.0, 100.0, 500.0, 'death']
SWEEP_ALPHA    = 0.25

penalty_results_025 = {}

for penalty in PENALTY_VALUES:
    is_death = penalty == 'death'
    ga_fn    = run_ga_death_penalty if is_death else run_ga_penalty_fitness
    p_label  = 'death' if is_death else f"x{penalty}"

    for init_method in ['uniform', 'biased']:
        label = (f"Penalty sweep a={SWEEP_ALPHA} "
                 f"{p_label} {init_method}")

        print(f"\n{'='*65}")
        print(f"  EXPERIMENT: {label}")
        print(f"{'='*65}")
        print(f"  Instance       : n={N}, m={M}, seed={INSTANCE_SEED}")
        print(f"  Tightness      : alpha={SWEEP_ALPHA}")
        print(f"  Penalty        : {p_label}")
        print(f"  Init method    : {init_method}")
        print(f"  Runs           : {N_RUNS}")
        print(f"  Offspring/run  : {N_OFFSPRING}")
        print(f"  Population     : {POP_SIZE}")
        print(f"  Mutation rate  : 1/n = {1/N:.4f}")
        print(f"  Tournament     : binary (k=2)")
        print(f"  Crossover      : uniform")
        print(f"  Reference best : {reference_bests[SWEEP_ALPHA]}")
        print(f"  p_j samples    : {PJ_SAMPLES:,}")
        print(f"  Thresholds     : {CONVERGENCE_THRESHOLDS}")
        print(f"{'='*65}")

        inst    = instances[SWEEP_ALPHA]
        v, w, c = inst['values'], inst['weights'], inst['capacities']
        ref     = reference_bests[SWEEP_ALPHA]
        pj      = pj_values[SWEEP_ALPHA]
        results = []

        for run in range(N_RUNS):
            print(f"  Run {run+1:02d}/{N_RUNS}...", end=" ", flush=True)

            base_kwargs = dict(
                n=N, m=M,
                values=v, weights=w, capacities=c,
                population_size=POP_SIZE,
                max_offspring=N_OFFSPRING,
                best_known=ref,
                track_every=TRACK_EVERY,
                init_method=init_method,
                p_j_values=pj if init_method == 'biased' else None,
                seed=run * 7,
            )
            if not is_death:
                base_kwargs['penalty_multiplier'] = penalty

            r = ga_fn(**base_kwargs)
            results.append(r)

            print(f"feas0={r['feasibility_rate_gen0']:.1%} "
                  f"div0={r['diversity_gen0']:.1f} "
                  f"final={r['pct_of_best_final']:.1f}%")

        if penalty not in penalty_results_025:
            penalty_results_025[penalty] = {}
        penalty_results_025[penalty][init_method] = results

        save_key  = label.replace(" ","_").replace("=","").replace(".","")
        save_path = f'/content/results_{save_key}.json'
        with open(save_path, 'w') as f:
            json.dump(make_serializable({
                'label':        label,
                'alpha':        SWEEP_ALPHA,
                'penalty':      str(penalty),
                'init_method':  init_method,
                'ref':          ref,
                'params': {
                    'n': N, 'm': M, 'instance_seed': INSTANCE_SEED,
                    'pop_size': POP_SIZE, 'n_runs': N_RUNS,
                    'n_offspring': N_OFFSPRING,
                    'mutation_rate': 1/N,
                    'penalty_mult': str(penalty),
                    'pj_samples': PJ_SAMPLES,
                    'pj_seed': PJ_SEED,
                    'ref_offspring': REF_OFFSPRING,
                    'ref_runs': REF_RUNS,
                    'tournament': 'binary k=2',
                    'crossover': 'uniform',
                    'convergence_thresholds': CONVERGENCE_THRESHOLDS,
                },
                'results': results,
            }), f, indent=2)
        print(f"  Saved -> {save_path}")
        files.download(save_path)

print("\nPenalty sweep alpha=0.25 complete.")
summarize_penalty_sweep(penalty_results_025, SWEEP_ALPHA,
                        reference_bests[SWEEP_ALPHA])


  EXPERIMENT: Penalty sweep a=0.25 x5.0 uniform
  Instance       : n=100, m=3, seed=42
  Tightness      : alpha=0.25
  Penalty        : x5.0
  Init method    : uniform
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 2375
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... feas0=0.0% div0=49.6 final=97.1%
  Run 02/20... feas0=0.0% div0=49.9 final=98.2%
  Run 03/20... feas0=0.0% div0=50.2 final=97.6%
  Run 04/20... feas0=0.0% div0=50.0 final=98.2%
  Run 05/20... feas0=0.0% div0=50.1 final=98.1%
  Run 06/20... feas0=0.0% div0=50.0 final=97.3%
  Run 07/20... feas0=0.0% div0=50.1 final=0.0%
  Run 08/20... feas0=0.0% div0=50.2 final=96.9%
  Run 09/20... feas0=0.0% div0=50.1 final=99.2%
  Run 10/20... feas0=0.0% div0=50.2 final=96.8%
  Run 11/20... feas0=0.0% div0=50.1 final=98.0%
  Run 12/20... feas0=0.0% div0=49.9 fi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  EXPERIMENT: Penalty sweep a=0.25 x5.0 biased
  Instance       : n=100, m=3, seed=42
  Tightness      : alpha=0.25
  Penalty        : x5.0
  Init method    : biased
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 2375
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... feas0=52.0% div0=19.1 final=95.6%
  Run 02/20... feas0=40.0% div0=19.0 final=97.8%
  Run 03/20... feas0=58.0% div0=19.1 final=99.1%
  Run 04/20... feas0=42.0% div0=19.1 final=0.0%
  Run 05/20... feas0=36.0% div0=19.0 final=0.0%
  Run 06/20... feas0=46.0% div0=18.9 final=96.9%
  Run 07/20... feas0=62.0% div0=19.0 final=97.0%
  Run 08/20... feas0=54.0% div0=19.0 final=98.7%
  Run 09/20... feas0=50.0% div0=19.0 final=97.9%
  Run 10/20... feas0=52.0% div0=19.1 final=96.0%
  Run 11/20... feas0=46.0% div0=19.1 final=0.0%
  Run 12/20... feas0=42.0% div0

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>


  EXPERIMENT: Penalty sweep a=0.25 death biased
  Instance       : n=100, m=3, seed=42
  Tightness      : alpha=0.25
  Penalty        : death
  Init method    : biased
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 2375
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... feas0=52.0% div0=19.1 final=98.0%
  Run 02/20... feas0=40.0% div0=19.0 final=96.4%
  Run 03/20... feas0=58.0% div0=19.1 final=96.0%
  Run 04/20... feas0=42.0% div0=19.1 final=97.4%
  Run 05/20... feas0=36.0% div0=19.0 final=93.9%
  Run 06/20... feas0=46.0% div0=18.9 final=97.4%
  Run 07/20... feas0=62.0% div0=19.0 final=95.7%
  Run 08/20... feas0=54.0% div0=19.0 final=95.4%
  Run 09/20... feas0=50.0% div0=19.0 final=96.4%
  Run 10/20... feas0=52.0% div0=19.1 final=93.1%
  Run 11/20... feas0=46.0% div0=19.1 final=98.2%
  Run 12/20... feas0=42.0%

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Penalty sweep alpha=0.25 complete.

  PENALTY SWEEP SUMMARY — alpha=0.25, uniform vs biased
  Reference best: 2375
  Metric                              Init    x5.0            x10.0           x25.0           x50.0           x100.0          x500.0          death           
  ----------------------------------------------------------------------------------------------------------------------
  Feasibility at gen 0                uniform 0.0%            0.0%            0.0%            0.0%            0.0%            0.0%            0.0%            
                                      biased  50.5%           50.5%           50.5%           50.5%           50.5%           50.5%           50.5%           

  Diversity at gen 0 (Hamming)        uniform 50.0            50.0            50.0            50.0            50.0            50.0            50.0            
                                      biased  19.0            19.0            19.0            19.0            19.0            

## 5b) Penalty Sweep, alpha=0.25

In [ ]:
# ============================================================
# PENALTY SWEEP — alpha=0.50
# penalties: 5, 10, 25, 50, 100, 500, death
# uniform vs biased, 20 runs, 20k offspring
# ============================================================
PENALTY_VALUES = [5.0, 10.0, 25.0, 50.0, 100.0, 500.0, 'death']
SWEEP_ALPHA = 0.50

penalty_results_050 = {}

for penalty in PENALTY_VALUES:
    is_death = penalty == 'death'
    ga_fn    = run_ga_death_penalty if is_death else run_ga_penalty_fitness
    p_label  = 'death' if is_death else f"x{penalty}"

    for init_method in ['uniform', 'biased']:
        label = (f"Penalty sweep a={SWEEP_ALPHA} "
                 f"{p_label} {init_method}")

        print(f"\n{'='*65}")
        print(f"  EXPERIMENT: {label}")
        print(f"{'='*65}")
        print(f"  Instance       : n={N}, m={M}, seed={INSTANCE_SEED}")
        print(f"  Tightness      : alpha={SWEEP_ALPHA}")
        print(f"  Penalty        : {p_label}")
        print(f"  Init method    : {init_method}")
        print(f"  Runs           : {N_RUNS}")
        print(f"  Offspring/run  : {N_OFFSPRING}")
        print(f"  Population     : {POP_SIZE}")
        print(f"  Mutation rate  : 1/n = {1/N:.4f}")
        print(f"  Tournament     : binary (k=2)")
        print(f"  Crossover      : uniform")
        print(f"  Reference best : {reference_bests[SWEEP_ALPHA]}")
        print(f"  p_j samples    : {PJ_SAMPLES:,}")
        print(f"  Thresholds     : {CONVERGENCE_THRESHOLDS}")
        print(f"{'='*65}")

        inst    = instances[SWEEP_ALPHA]
        v, w, c = inst['values'], inst['weights'], inst['capacities']
        ref     = reference_bests[SWEEP_ALPHA]
        pj      = pj_values[SWEEP_ALPHA]
        results = []

        for run in range(N_RUNS):
            print(f"  Run {run+1:02d}/{N_RUNS}...", end=" ", flush=True)

            base_kwargs = dict(
                n=N, m=M,
                values=v, weights=w, capacities=c,
                population_size=POP_SIZE,
                max_offspring=N_OFFSPRING,
                best_known=ref,
                track_every=TRACK_EVERY,
                init_method=init_method,
                p_j_values=pj if init_method == 'biased' else None,
                seed=run * 7,
            )
            if not is_death:
                base_kwargs['penalty_multiplier'] = penalty

            r = ga_fn(**base_kwargs)
            results.append(r)

            print(f"feas0={r['feasibility_rate_gen0']:.1%} "
                  f"div0={r['diversity_gen0']:.1f} "
                  f"final={r['pct_of_best_final']:.1f}%")

        if penalty not in penalty_results_050:
            penalty_results_050[penalty] = {}
        penalty_results_050[penalty][init_method] = results

        save_key  = label.replace(" ","_").replace("=","").replace(".","")
        save_path = f'/content/results_{save_key}.json'
        with open(save_path, 'w') as f:
            json.dump(make_serializable({
                'label':        label,
                'alpha':        SWEEP_ALPHA,
                'penalty':      str(penalty),
                'init_method':  init_method,
                'ref':          ref,
                'params': {
                    'n': N, 'm': M, 'instance_seed': INSTANCE_SEED,
                    'pop_size': POP_SIZE, 'n_runs': N_RUNS,
                    'n_offspring': N_OFFSPRING,
                    'mutation_rate': 1/N,
                    'penalty_mult': str(penalty),
                    'pj_samples': PJ_SAMPLES,
                    'pj_seed': PJ_SEED,
                    'ref_offspring': REF_OFFSPRING,
                    'ref_runs': REF_RUNS,
                    'tournament': 'binary k=2',
                    'crossover': 'uniform',
                    'convergence_thresholds': CONVERGENCE_THRESHOLDS,
                },
                'results': results,
            }), f, indent=2)
        print(f"  Saved -> {save_path}")
        files.download(save_path)

print("\nPenalty sweep alpha=0.50 complete.")
summarize_penalty_sweep(penalty_results_050, SWEEP_ALPHA,
                        reference_bests[SWEEP_ALPHA])


  EXPERIMENT: Penalty sweep a=0.5 x5.0 uniform
  Instance       : n=100, m=3, seed=42
  Tightness      : alpha=0.5
  Penalty        : x5.0
  Init method    : uniform
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 3909
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... feas0=30.0% div0=49.6 final=99.7%
  Run 02/20... feas0=48.0% div0=49.9 final=99.2%
  Run 03/20... feas0=40.0% div0=50.2 final=99.4%
  Run 04/20... feas0=36.0% div0=50.0 final=99.8%
  Run 05/20... feas0=36.0% div0=50.1 final=0.0%
  Run 06/20... feas0=42.0% div0=50.0 final=99.6%
  Run 07/20... feas0=26.0% div0=50.1 final=99.1%
  Run 08/20... feas0=28.0% div0=50.2 final=98.8%
  Run 09/20... feas0=36.0% div0=50.1 final=98.6%
  Run 10/20... feas0=36.0% div0=50.2 final=99.7%
  Run 11/20... feas0=32.0% div0=50.1 final=98.1%
  Run 12/20... feas0=28.0% di

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  EXPERIMENT: Penalty sweep a=0.5 x5.0 biased
  Instance       : n=100, m=3, seed=42
  Tightness      : alpha=0.5
  Penalty        : x5.0
  Init method    : biased
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 3909
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... feas0=76.0% div0=49.4 final=99.6%
  Run 02/20... feas0=72.0% div0=49.8 final=99.6%
  Run 03/20... feas0=70.0% div0=49.7 final=99.4%
  Run 04/20... feas0=72.0% div0=49.8 final=0.0%
  Run 05/20... feas0=68.0% div0=49.5 final=99.6%
  Run 06/20... feas0=66.0% div0=49.5 final=99.1%
  Run 07/20... feas0=70.0% div0=49.6 final=99.4%
  Run 08/20... feas0=58.0% div0=49.6 final=99.1%
  Run 09/20... feas0=82.0% div0=49.2 final=0.0%
  Run 10/20... feas0=76.0% div0=49.7 final=98.3%
  Run 11/20... feas0=74.0% div0=49.7 final=97.9%
  Run 12/20... feas0=62.0% div0=

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>


  EXPERIMENT: Penalty sweep a=0.5 death biased
  Instance       : n=100, m=3, seed=42
  Tightness      : alpha=0.5
  Penalty        : death
  Init method    : biased
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 3909
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... feas0=76.0% div0=49.4 final=98.6%
  Run 02/20... feas0=72.0% div0=49.8 final=98.4%
  Run 03/20... feas0=70.0% div0=49.7 final=99.3%
  Run 04/20... feas0=72.0% div0=49.8 final=98.6%
  Run 05/20... feas0=68.0% div0=49.5 final=99.8%
  Run 06/20... feas0=66.0% div0=49.5 final=98.1%
  Run 07/20... feas0=70.0% div0=49.6 final=99.2%
  Run 08/20... feas0=58.0% div0=49.6 final=98.4%
  Run 09/20... feas0=82.0% div0=49.2 final=97.7%
  Run 10/20... feas0=76.0% div0=49.7 final=98.6%
  Run 11/20... feas0=74.0% div0=49.7 final=98.1%
  Run 12/20... feas0=62.0% d

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Penalty sweep alpha=0.50 complete.

  PENALTY SWEEP SUMMARY — alpha=0.5, uniform vs biased
  Reference best: 3909
  Metric                              Init    x5.0            x10.0           x25.0           x50.0           x100.0          x500.0          death           
  ----------------------------------------------------------------------------------------------------------------------
  Feasibility at gen 0                uniform 32.6%           32.6%           32.6%           32.6%           32.6%           32.6%           32.6%           
                                      biased  71.8%           71.8%           71.8%           71.8%           71.8%           71.8%           71.8%           

  Diversity at gen 0 (Hamming)        uniform 50.0            50.0            50.0            50.0            50.0            50.0            50.0            
                                      biased  49.5            49.5            49.5            49.5            49.5            4



## Experiment 6 — Tightness Sweep
*Death penalty | uniform vs biased vs 50% hybrid | alpha 0.10 to 0.90*
*Run Cell 6a (sweep) — Cell 6b (summary) runs automatically at end*
*Note: alpha=0.10 and 0.20 have p_j=0 due to Monte Carlo precision limits at extreme tightness — biased degenerates to empty knapsack at those levels, results noted as limitation*

In [ ]:
# ============================================================
# TIGHTNESS SWEEP — death penalty
# alpha: 0.10 to 0.90
# methods: uniform, biased, 50% hybrid
# 20 runs, 20k offspring
# ============================================================
TIGHTNESS_ALPHAS  = [0.10, 0.20, 0.25, 0.30, 0.40, 0.50,
                     0.60, 0.70, 0.75, 0.80, 0.90]
HYBRID_RATIO      = 0.50
TIGHTNESS_METHODS = ['uniform', 'biased', 'hybrid']

tightness_results = {}

for alpha in TIGHTNESS_ALPHAS:
    tightness_results[alpha] = {}
    inst    = instances[alpha]
    v, w, c = inst['values'], inst['weights'], inst['capacities']
    ref     = reference_bests[alpha]
    pj      = pj_values[alpha]

    for init_method in TIGHTNESS_METHODS:
        label = (f"Tightness sweep death "
                 f"a={alpha} {init_method}"
                 + (f" r={HYBRID_RATIO}" if init_method == 'hybrid' else ""))

        print(f"\n{'='*65}")
        print(f"  EXPERIMENT: {label}")
        print(f"{'='*65}")
        print(f"  Instance       : n={N}, m={M}, seed={INSTANCE_SEED}")
        print(f"  Tightness      : alpha={alpha}")
        print(f"  Penalty        : death")
        print(f"  Init method    : {init_method}" +
              (f" (ratio={HYBRID_RATIO})" if init_method == 'hybrid' else ""))
        print(f"  Runs           : {N_RUNS}")
        print(f"  Offspring/run  : {N_OFFSPRING}")
        print(f"  Population     : {POP_SIZE}")
        print(f"  Mutation rate  : 1/n = {1/N:.4f}")
        print(f"  Tournament     : binary (k=2)")
        print(f"  Crossover      : uniform")
        print(f"  Reference best : {ref}")
        print(f"  p_j samples    : {PJ_SAMPLES:,}")
        print(f"  Thresholds     : {CONVERGENCE_THRESHOLDS}")
        print(f"{'='*65}")

        results = []

        for run in range(N_RUNS):
            print(f"  Run {run+1:02d}/{N_RUNS}...", end=" ", flush=True)

            r = run_ga_death_penalty(
                n=N, m=M,
                values=v, weights=w, capacities=c,
                population_size=POP_SIZE,
                max_offspring=N_OFFSPRING,
                best_known=ref,
                track_every=TRACK_EVERY,
                init_method=init_method,
                p_j_values=pj if init_method != 'uniform' else None,
                hybrid_ratio=HYBRID_RATIO if init_method == 'hybrid' else None,
                seed=run * 7,
            )
            results.append(r)

            print(f"feas0={r['feasibility_rate_gen0']:.1%} "
                  f"div0={r['diversity_gen0']:.1f} "
                  f"final={r['pct_of_best_final']:.1f}%")

        tightness_results[alpha][init_method] = results

        save_key  = label.replace(" ","_").replace("=","").replace(".","")
        save_path = f'/content/results_{save_key}.json'
        with open(save_path, 'w') as f:
            json.dump(make_serializable({
                'label':        label,
                'alpha':        alpha,
                'penalty':      'death',
                'init_method':  init_method,
                'hybrid_ratio': HYBRID_RATIO if init_method == 'hybrid' else None,
                'ref':          ref,
                'params': {
                    'n': N, 'm': M, 'instance_seed': INSTANCE_SEED,
                    'pop_size': POP_SIZE, 'n_runs': N_RUNS,
                    'n_offspring': N_OFFSPRING,
                    'mutation_rate': 1/N,
                    'penalty_mult': 'death',
                    'pj_samples': PJ_SAMPLES,
                    'pj_seed': PJ_SEED,
                    'ref_offspring': REF_OFFSPRING,
                    'ref_runs': REF_RUNS,
                    'tournament': 'binary k=2',
                    'crossover': 'uniform',
                    'convergence_thresholds': CONVERGENCE_THRESHOLDS,
                    'hybrid_ratio': HYBRID_RATIO,
                },
                'results': results,
            }), f, indent=2)
        print(f"\n  Saved -> {save_path}")

print("\nTightness sweep complete.")
summarize_tightness_sweep(tightness_results)

# Miscallaneous

In [ ]:
# ============================================================
# Clean Print Summary for Results.md
# ============================================================
import json
from datetime import date

CONVERGENCE_THRESHOLDS = [50, 60, 70, 80, 90, 95, 98, 99]
N, M, INSTANCE_SEED    = 100, 3, 42
POP_SIZE               = 50
N_RUNS                 = 20
N_OFFSPRING            = 20000
PJ_SAMPLES             = 10_000_000
PJ_SEED                = 42
REF_OFFSPRING          = 30_000
REF_RUNS               = 20
PENALTY_MULT           = 5.0

with open('reference_bests.json') as f:
    ref_meta = json.load(f)
reference_bests = {float(k): v for k, v in
                   ref_meta['reference_bests'].items()}

conditions = [
    ('results_Death_penalty_α025.json', 'Death penalty a=0.25'),
    ('results_Death_penalty_α050.json', 'Death penalty a=0.50'),
    ('results_Death_penalty_α075.json', 'Death penalty a=0.75'),
    ('results_Penalty_x50__α025.json',  'Penalty x5.0 a=0.25'),
    ('results_Penalty_x50__α050.json',  'Penalty x5.0 a=0.50'),
    ('results_Penalty_x50__α075.json',  'Penalty x5.0 a=0.75'),
]

def mean_field(runs, field):
    vals = [r[field] for r in runs if r.get(field) is not None]
    return sum(vals)/len(vals) if vals else 0.0

def thresh_stats(runs, t):
    vals = []
    for r in runs:
        c = r.get('convergence', {})
        v = c.get(str(t)) if c.get(str(t)) is not None else c.get(t)
        if v is not None:
            vals.append(v)
    if not vals:
        return 'never', 0
    return f"{sum(vals)/len(vals):.0f}", len(vals)

lines = []
lines.append("# Baseline Experiment Results")
lines.append(f"\nGenerated: {date.today()}")
lines.append("\n## Experiment Parameters\n")
lines.append("| Parameter | Value |")
lines.append("|---|---|")
lines.append(f"| n (items) | {N} |")
lines.append(f"| m (constraints) | {M} |")
lines.append(f"| Instance seed | {INSTANCE_SEED} |")
lines.append(f"| Population size | {POP_SIZE} |")
lines.append(f"| Runs per condition | {N_RUNS} |")
lines.append(f"| Offspring per run | {N_OFFSPRING} |")
lines.append(f"| Mutation rate | 1/n = {1/N:.4f} |")
lines.append(f"| Tournament | binary k=2 |")
lines.append(f"| Crossover | uniform |")
lines.append(f"| p_j samples | {PJ_SAMPLES:,} |")
lines.append(f"| Convergence thresholds | {CONVERGENCE_THRESHOLDS} |")

lines.append("\n## Reference Bests\n")
lines.append(
    f"Established via repair-based GA, {REF_RUNS} runs, "
    f"{REF_OFFSPRING:,} offspring, seeds 0-{REF_RUNS-1}. "
    "Not proven optimal -- best known values used as convergence targets.\n"
)
lines.append("| Alpha | Reference Best |")
lines.append("|---|---|")
for alpha, ref in sorted(reference_bests.items()):
    lines.append(f"| {alpha} | {ref} |")

lines.append("\n## p_j Summary\n")
lines.append("| Alpha | p_j min | p_j max | p_j mean | Interpretation |")
lines.append("|---|---|---|---|---|")
lines.append("| 0.25 | 0.000 | 1.000 | 0.240 | Strong differentiation -- tight instance |")
lines.append("| 0.50 | 0.414 | 0.493 | 0.449 | Weak differentiation -- medium instance |")
lines.append("| 0.75 | 0.500 | 0.500 | 0.500 | No differentiation -- degenerates to uniform |")

lines.append("\n## Results by Condition\n")

for fname, label in conditions:
    with open(fname) as f:
        d = json.load(f)

    ref  = d['ref']
    uni  = d['uniform']
    bia  = d['biased']
    n_runs = len(uni)

    lines.append(f"### {label}\n")
    lines.append(f"Reference best: {ref}  |  Runs: {n_runs}  "
                 f"|  Offspring: {d['params']['n_offspring']}\n")
    lines.append("| Metric | Uniform | Biased |")
    lines.append("|---|---|---|")
    lines.append(
        f"| Feasibility at gen 0 "
        f"| {mean_field(uni,'feasibility_rate_gen0'):.1%} "
        f"| {mean_field(bia,'feasibility_rate_gen0'):.1%} |"
    )
    lines.append(
        f"| Fitness at gen 0 (% of ref) "
        f"| {mean_field(uni,'pct_of_best_gen0'):.1f}% "
        f"| {mean_field(bia,'pct_of_best_gen0'):.1f}% |"
    )
    lines.append(
        f"| Final fitness (% of ref) "
        f"| {mean_field(uni,'pct_of_best_final'):.1f}% "
        f"| {mean_field(bia,'pct_of_best_final'):.1f}% |"
    )
    for t in CONVERGENCE_THRESHOLDS:
        u_mean, u_count = thresh_stats(uni, t)
        b_mean, b_count = thresh_stats(bia, t)
        u_str = f"{u_mean} ({u_count}/{n_runs})" if u_mean != 'never' else 'never'
        b_str = f"{b_mean} ({b_count}/{n_runs})" if b_mean != 'never' else 'never'
        lines.append(f"| Offspring to {t}% | {u_str} | {b_str} |")

    lines.append("")

md = '\n'.join(lines)
with open('RESULTS.md', 'w') as f:
    f.write(md)
print("RESULTS.md written")

from google.colab import files
files.download('RESULTS.md')

RESULTS.md written


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# ============================================================
# RERUN — 4 missing penalty sweep conditions
# ============================================================
missing_conditions = [
    (0.25, 25.0,  'uniform'),
    (0.25, 500.0, 'biased'),
    (0.25, 500.0, 'uniform'),
    (0.50, 'death', 'uniform'),
]

for alpha, penalty, init_method in missing_conditions:
    is_death = penalty == 'death'
    ga_fn    = run_ga_death_penalty if is_death else run_ga_penalty_fitness
    p_label  = 'death' if is_death else f"x{penalty}"
    label    = f"Penalty sweep a={alpha} {p_label} {init_method}"

    print(f"\n{'='*65}")
    print(f"  EXPERIMENT: {label}")
    print(f"{'='*65}")
    print(f"  Penalty : {p_label} | Init : {init_method} | Alpha : {alpha}")
    print(f"{'='*65}")

    inst    = instances[alpha]
    v, w, c = inst['values'], inst['weights'], inst['capacities']
    ref     = reference_bests[alpha]
    pj      = pj_values[alpha]
    results = []

    for run in range(N_RUNS):
        print(f"  Run {run+1:02d}/{N_RUNS}...", end=" ", flush=True)

        base_kwargs = dict(
            n=N, m=M,
            values=v, weights=w, capacities=c,
            population_size=POP_SIZE,
            max_offspring=N_OFFSPRING,
            best_known=ref,
            track_every=TRACK_EVERY,
            init_method=init_method,
            p_j_values=pj if init_method == 'biased' else None,
            seed=run * 7,
        )
        if not is_death:
            base_kwargs['penalty_multiplier'] = penalty

        r = ga_fn(**base_kwargs)
        results.append(r)
        print(f"feas0={r['feasibility_rate_gen0']:.1%} "
              f"div0={r['diversity_gen0']:.1f} "
              f"final={r['pct_of_best_final']:.1f}%")

    save_key  = label.replace(" ","_").replace("=","").replace(".","")
    save_path = f'/content/results_{save_key}.json'
    with open(save_path, 'w') as f:
        json.dump(make_serializable({
            'label':        label,
            'alpha':        alpha,
            'penalty':      str(penalty),
            'init_method':  init_method,
            'ref':          ref,
            'params': {
                'n': N, 'm': M, 'instance_seed': INSTANCE_SEED,
                'pop_size': POP_SIZE, 'n_runs': N_RUNS,
                'n_offspring': N_OFFSPRING,
                'mutation_rate': 1/N,
                'penalty_mult': str(penalty),
                'pj_samples': PJ_SAMPLES,
                'pj_seed': PJ_SEED,
                'ref_offspring': REF_OFFSPRING,
                'ref_runs': REF_RUNS,
                'tournament': 'binary k=2',
                'crossover': 'uniform',
                'convergence_thresholds': CONVERGENCE_THRESHOLDS,
            },
            'results': results,
        }), f, indent=2)
    print(f"  Saved -> {save_path}")

print("\nAll missing conditions complete.")
print("Download these 4 files:")
print("  results_Penalty_sweep_a025_x250_uniform.json")
print("  results_Penalty_sweep_a025_x5000_biased.json")
print("  results_Penalty_sweep_a025_x5000_uniform.json")
print("  results_Penalty_sweep_a05_death_uniform.json")


  EXPERIMENT: Penalty sweep a=0.25 x25.0 uniform
  Penalty : x25.0 | Init : uniform | Alpha : 0.25
  Run 01/20... feas0=0.0% div0=49.6 final=97.2%
  Run 02/20... feas0=0.0% div0=49.9 final=99.7%
  Run 03/20... feas0=0.0% div0=50.2 final=98.7%
  Run 04/20... feas0=0.0% div0=50.0 final=97.7%
  Run 05/20... feas0=0.0% div0=50.1 final=97.3%
  Run 06/20... feas0=0.0% div0=50.0 final=96.1%
  Run 07/20... feas0=0.0% div0=50.1 final=96.8%
  Run 08/20... feas0=0.0% div0=50.2 final=95.0%
  Run 09/20... feas0=0.0% div0=50.1 final=94.9%
  Run 10/20... feas0=0.0% div0=50.2 final=96.5%
  Run 11/20... feas0=0.0% div0=50.1 final=98.8%
  Run 12/20... feas0=0.0% div0=49.9 final=95.8%
  Run 13/20... feas0=0.0% div0=50.1 final=95.5%
  Run 14/20... feas0=0.0% div0=49.6 final=96.5%
  Run 15/20... feas0=0.0% div0=50.0 final=97.4%
  Run 16/20... feas0=0.0% div0=50.1 final=97.7%
  Run 17/20... feas0=0.0% div0=50.0 final=96.5%
  Run 18/20... feas0=0.0% div0=50.2 final=96.8%
  Run 19/20... feas0=0.0% div0=50.1 